Initial thoughts/findings:
* What should we do about low sample numbers for: Northern Ireland; North East England; Wales

In [ ]:
# imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# import plotly.io as pio
# pio.renderers.default = "notebook"

from dap_job_quality.getters.healthcare import *
from dap_job_quality.analysis.healthcare_analysis.utils import create_uk_heatmap, plot_years, grouped_bar, stacked_bar

In [ ]:
# data
healthcare_sample = get_salaries_w_soc_region()
job_titles = get_occupations()

healthcare_sample = pd.read_parquet("s3://open-jobs-lake/job_quality/health_social_care/outputs/healthcare_sample.parquet")
    
sample_w_salaries_dimensions = pd.read_parquet("s3://open-jobs-lake/job_quality/health_social_care/outputs/sample_w_salaries_dimensions.parquet")
    
contracts_salaries_soc_region = pd.read_parquet("s3://open-jobs-lake/job_quality/health_social_care/outputs/contracts_salaries_soc_region.parquet")

In [ ]:
pd.DataFrame(healthcare_sample['job_title_raw_x'].value_counts()).reset_index().to_csv('job_titles.csv', index=False)

In [ ]:
healthcare_sample[healthcare_sample['job_title_raw_x']=='RMN']['description'].to_csv('rmn.csv', index=False)

In [ ]:
sample_w_salaries_dimensions['soc_4_digit_name'].value_counts()

In [ ]:
pd.DataFrame(sample_w_salaries_dimensions.groupby(['soc_4_digit_name', 'year']).size()).to_csv('sample_size_by_year.csv')

In [ ]:
def get_groups_w_valid_sample_size(df, grouping_cols=['soc_4_digit_name', 'year'], min_sample_size=100):
    if isinstance(grouping_cols, str):
        grouping_cols = [grouping_cols]
        
    # Perform groupby and count
    grouped_counts = df.groupby(grouping_cols).size()

    # If grouping by more than one column, unstack to create a wide format
    if len(grouping_cols) > 1:
        grouped_counts = grouped_counts.unstack()

        # Find groups where all years have count >= min_sample_size
        valid_groups = grouped_counts[(grouped_counts >= min_sample_size).all(axis=1)].index.tolist()
    else:
        # If only one column is used for grouping, check directly
        valid_groups = grouped_counts[grouped_counts >= min_sample_size].index.tolist()

    return valid_groups
    
    # grouped_counts = df.groupby(grouping_cols).size().unstack()

    # valid_soc_names = grouped_counts[(grouped_counts >= min_sample_size).all(axis=1)].index.tolist()
    
    # return valid_soc_names

valid_soc_names = get_groups_w_valid_sample_size(sample_w_salaries_dimensions)
valid_soc_names


In [ ]:
contracts_salaries_soc_region

In [ ]:
contracts_salaries_soc_region.columns

In [ ]:
# contract_counts = contracts_salaries_soc_region.groupby(['soc_4_digit_name']).size().reset_index(name='count')
# contract_counts

contract_socs = get_groups_w_valid_sample_size(contracts_salaries_soc_region, ['soc_4_digit_name'], 100)
contract_socs

In [ ]:
contract_sample = contracts_salaries_soc_region[contracts_salaries_soc_region['soc_4_digit_name'].isin(contract_socs)]

contract_counts = contract_sample.groupby(['soc_4_digit_name', 'final_contract_type']).size().reset_index(name='count')
contract_counts['percentage'] = contract_counts.groupby('soc_4_digit_name')['count'].transform(lambda x: x / x.sum() * 100)

contract_counts

In [ ]:
contract_sample['soc_4_digit_name'].value_counts().index

In [ ]:
contract_sample[['id','final_contract_type',
       'itl_1_name','soc_4_digit_name', 'year',
       ]]

In [ ]:
df_counts = contract_sample.groupby(["soc_4_digit_name", "itl_1_name", "final_contract_type"]).size().unstack(fill_value=0)
df_counts

In [ ]:
import plotly.graph_objects as go

# Step 1: Aggregate data to get proportions
df_counts = contract_sample.groupby(["soc_4_digit_name", "itl_1_name", "final_contract_type"]).size().reset_index(name="count")#.unstack(fill_value=0)

# # Compute proportion of Permanent contracts
# df_counts["permanent_proportion"] = df_counts["Permanent"] / (df_counts["Permanent"] + df_counts["Temporary"])

# # Reset index for easier handling
# df_counts = df_counts.reset_index()

# Step 2: Get unique soc_4_digit_name values
unique_socs = df_counts["soc_4_digit_name"].unique()

# Step 3: Create an interactive Plotly figure
fig = go.Figure()

# Step 4: Add traces for each soc_4_digit_name (initially hidden)
for i, soc in enumerate(unique_socs):
    filtered_data = df_counts[df_counts["soc_4_digit_name"] == soc]
    
    # Create traces for each contract type (stacked)
    traces = []
    for contract_type in df_counts["final_contract_type"].unique():
        sub_data = filtered_data[filtered_data["final_contract_type"] == contract_type]
        
        trace = go.Bar(
            x=sub_data["count"],
            y=sub_data["itl_1_name"],
            name=contract_type,
            orientation="h",
            visible=(i == 0)  # Only show the first SOC 4-digit name by default
        )
        traces.append(trace)
    
    fig.add_traces(traces)
    
    # trace = go.Bar(
    #     x=filtered_data["permanent_proportion"],
    #     y=filtered_data["itl_1_name"],
    #     orientation="h",
    #     name=soc,
    #     visible=(i == 0)  # Only show the first trace by default
    # )
    
    # fig.add_trace(trace)

# Step 5: Create dropdown menu
dropdown_buttons = [
    {
        "label": soc,
        "method": "update",
        "args": [
            {"visible": [(j // len(df_counts["final_contract_type"].unique())) == i for j in range(len(fig.data))]},
        ],
    }
    for i, soc in enumerate(unique_socs)
]

# Step 6: Update layout with dropdown
fig.update_layout(
    updatemenus=[{
        "buttons": dropdown_buttons,
        "direction": "down",
        "showactive": True,
        "x": 0.5,  # Center the dropdown horizontally
        "y": 1.1,  # Move dropdown above the chart
        "xanchor": "center",  # Anchor horizontally at center
        "yanchor": "top"  # Anchor vertically at the top
    }],
    title=None,#"Proportion of Permanent Contracts by ITL1 Region",
    xaxis_title="Number of job adverts",
    yaxis_title="ITL1 Region",
    width=1200,
    height=800
)

fig.show()

In [ ]:
contract_sample.columns

contracts_and_benefits = pd.merge(contract_sample, sample_w_salaries_dimensions, on='id', how='inner')
contracts_and_benefits.drop(columns=['soc_4_digit_name_y'], inplace=True)
contracts_and_benefits.rename(columns={'soc_4_digit_name_x':'soc_4_digit_name'}, inplace=True)
len(contracts_and_benefits) - len(contract_sample)

In [ ]:
contracts_and_benefits.columns

In [ ]:


# Sample aggregation: compute proportion of FLEX_HOURS == 1
grouped_flex = contracts_and_benefits.groupby(["soc_4_digit_name", "final_contract_type"])["FLEX_HOURS"].mean().reset_index()

# Rename the column for clarity
grouped_flex.rename(columns={"FLEX_HOURS": "flex_hours_proportion"}, inplace=True)

grouped_flex


In [ ]:
fig = px.bar(
    grouped_flex,
    x="flex_hours_proportion",
    y="soc_4_digit_name",
    color="final_contract_type",
    orientation="h",
    title="Proportion of Job Adverts Offering Flexible Hours",
    labels={"flex_hours_proportion": "Proportion of Flexible Hours Ads", "soc_4_digit_name": "SOC 4-Digit Name"},
    barmode="group",  # Side-by-side bars
    width=1000,
    height=800
)

# Show the chart
fig.show()

In [ ]:
# Compute total job ads and number of ads with FLEX_HOURS == 1
grouped_flex = contracts_and_benefits.groupby(["soc_4_digit_name", "final_contract_type"]).agg(
    total_ads=("FLEX_HOURS", "count"),
    flex_hours_ads=("FLEX_HOURS", "sum")  # Sum of 1s gives count of flexible hour jobs
).reset_index()

# Compute proportion
grouped_flex["flex_hours_proportion"] = grouped_flex["flex_hours_ads"] / grouped_flex["total_ads"]

grouped_flex


In [ ]:
import plotly.figure_factory as ff
import numpy as np

# Pivot table for heatmap format
heatmap_data = grouped_flex.pivot(index="soc_4_digit_name", columns="final_contract_type", values="flex_hours_proportion")
text_data = grouped_flex.pivot(index="soc_4_digit_name", columns="final_contract_type", values="total_ads")

# Convert DataFrame to NumPy arrays
z_values = heatmap_data.values  # Color values (proportion)
text_values = text_data.values  # Display values (counts)

# Create heatmap figure
fig = ff.create_annotated_heatmap(
    z=z_values,  # Color based on proportion
    x=heatmap_data.columns.tolist(),  # Contract Types (Columns)
    y=heatmap_data.index.tolist(),  # SOC 4-digit Names (Rows)
    annotation_text=text_values,  # Display absolute counts
    colorscale="Blues",  # Use a blue color scale
    showscale=True  # Show color scale legend
)

# Update layout
fig.update_layout(
    title="Flexible Hours by SOC 4-Digit Name and Contract Type",
    xaxis_title="Contract Type",
    yaxis_title="SOC 4-Digit Name",
    width=1000,
    height=800
)

# Show the figure
fig.show()

In [ ]:
contract_sample[contract_sample['soc_4_digit_name']=='Other registered nursing professionals']['job_title_raw_x'].value_counts()

In [ ]:
contract_sample[(contract_sample['soc_4_digit_name']=='Other registered nursing professionals') & (contract_sample['job_title_raw_x']=='SEN TA')][['id', 'description']]

In [ ]:
contract_sample.columns

In [ ]:
sorted_order = (
            contract_counts[contract_counts["final_contract_type"] == 'Permanent']
            .sort_values(by="percentage", ascending=False)["soc_4_digit_name"]
            .tolist()
        )

sorted_order

In [ ]:
stacked_bar(contract_counts, x='percentage', y='soc_4_digit_name', colour='final_contract_type', sorted_order=sorted_order)

In [ ]:
healthcare_sample = get_salaries_w_soc_region()

In [ ]:
healthcare_sample.columns

In [ ]:
healthcare_sample['year'] = pd.to_datetime(healthcare_sample['created_x'], dayfirst=True).dt.year

healthcare_sample['year'].value_counts()

In [ ]:
len(healthcare_sample[healthcare_sample['year']>2020])

In [ ]:
healthcare_sample = healthcare_sample[healthcare_sample['year']>2020]

healthcare_sample['year'].value_counts()

In [ ]:
year_counts = healthcare_sample['year'].value_counts().reset_index()
year_counts.columns = ['year', 'count']

# Sort by year for a better visual order
year_counts = year_counts.sort_values(by="year")

# Create a bar chart
fig = px.bar(year_counts, x='year', y='count', 
             text='count', title="Number of Healthcare-related Job Adverts per Year",
             labels={'year': 'Year', 'count': 'Count'},
             template="plotly_white")

# Display count labels on bars
fig.update_traces(textposition='outside')

fig.update_layout(
    xaxis=dict(
        type='category',  # Treats years as discrete categories (ensures no decimals)
        tickmode='array',  # Sets specific ticks
        tickvals=year_counts['year'],  # Only show actual years
        tickformat='d'  # Ensures numbers are displayed as whole numbers
    )
)

# Show the plot
fig.show()

# Sample size

In [ ]:
job_titles = get_occupations()
job_titles.head()

In [ ]:
grouped = job_titles.groupby("soc_4_digit_name")["Group Title"].unique().reset_index()
grouped

In [ ]:
# Count occurrences of each soc_4_digit_name
soc_counts = healthcare_sample['soc_4_digit_name'].value_counts().reset_index()
soc_counts.columns = ['soc_4_digit_name', 'count']

# Create the bar chart
fig = px.bar(soc_counts, 
             y='soc_4_digit_name', 
             x='count', 
            #  title='Counts of SOC 4-Digit Names',
             labels={'soc_4_digit_name': 'SOC 4-Digit Name', 'count': 'Count'},
             text='count')

# Customize layout
fig.update_layout(width=1000,
                  height=600,
                  xaxis={'categoryorder':'total descending'},
                  xaxis_tickangle=-45)  # Rotate labels for better readability

fig.show()

In [ ]:
healthcare_sample['itl_1_code']#.columns#['itl_1_name'].value_counts()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Load the ITL1 shapefile (update with actual file path)
shapefile_path = "ITL1.geojson"
gdf = gpd.read_file(shapefile_path)

counts = healthcare_sample['itl_1_name'].value_counts()

df = pd.DataFrame(healthcare_sample['itl_1_code'].value_counts())

gdf = gdf.merge(df, left_on="ITL121CD", right_on="itl_1_code", how="left")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
gdf.plot(column="count", cmap="viridis", linewidth=0.8, edgecolor="black", legend=True, ax=ax)

for _, row in gdf.iterrows():
    if row["geometry"].centroid:  # Check if centroid exists
        x, y = row["geometry"].centroid.x, row["geometry"].centroid.y
        ax.text(x, y, f"{int(row['count'])}", color="white", fontsize=10,
                ha="center", va="center", fontweight="bold", bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', boxstyle="round,pad=0.3"))

ax.set_title("Heatmap of ITL1 Regions", fontsize=14)
plt.show()


In [ ]:
counts

In [ ]:
def two_way_heatmap(df, var1 = 'soc_4_digit_name', var2 = 'itl_1_name'):
    # Create a contingency table (cross-tabulation) of counts
    heatmap_data = df.groupby([var1, var2]).size().reset_index(name='count')

    # Create the heatmap
    fig = px.density_heatmap(
        heatmap_data, 
        x=var2, 
        y=var1, 
        z='count', 
        histfunc='sum',
        # title="SOC 4-Digit Name vs ITL 1 Name Heatmap",
        # labels={'itl_1_name': 'ITL 1 Name', 'soc_4_digit_name': 'SOC 4-Digit Name', 'count': 'Count'},
        color_continuous_scale='Viridis',
        text_auto=True,
    )

    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        xaxis_tickangle=-45  # Rotate x-axis labels for readability
    )

    return fig

In [ ]:
two_way_heatmap(healthcare_sample, var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

In [ ]:
top_professions = ['Other registered nursing professionals',
                   'Other health professionals n.e.c.',
                   'Registered mental health nurses',
                   'Occupational therapists']

In [ ]:
two_way_heatmap(healthcare_sample[healthcare_sample['soc_4_digit_name'].isin(top_professions)], var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

In [ ]:
two_way_heatmap(healthcare_sample[~healthcare_sample['soc_4_digit_name'].isin(top_professions)], var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

# Pay by 4 digit SOC code

In [ ]:
# salary boxplot
fig = px.box(healthcare_sample, y='soc_4_digit_name', x='hourly_wage')

fig.update_layout(width=1000,
                  height=600,
                  xaxis={'categoryorder':'total descending'},
                  )

fig.show()

In [ ]:
healthcare_sample['soc_4_digit_name'].unique()

In [ ]:
nursing_professions = ['Registered community nurses',
       'Other registered nursing professionals',
       'Registered mental health nurses',
       'Registered specialist nurses',
       'Registered nurse practitioners', 'Midwifery nurses',
       "Registered children's nurses"
       ]

In [ ]:
fig = px.box(healthcare_sample[healthcare_sample['soc_4_digit_name'].isin(nursing_professions)], y='soc_4_digit_name', x='hourly_wage')

fig.update_layout(width=1000,
                  height=600,
                  xaxis={'categoryorder':'total descending'},
                  )

fig.show()

# Job quality data

In [ ]:
job_qual = pd.read_parquet('s3://open-jobs-lake/job_quality/health_social_care/health_jobs_jq_dimensions.parquet')

In [ ]:
job_qual.head()

In [ ]:
job_qual_processed, dimensions_wide = analysis_utils.process_jq_data(job_qual)
job_qual_processed.head()

In [ ]:
dimensions_wide

In [ ]:
sample_w_salaries_dimensions = pd.merge(healthcare_sample, dimensions_wide, on='id', how='left')
sample_w_salaries_dimensions.head()

In [ ]:
sample_w_salaries_dimensions.columns

In [ ]:
prop_table = sample_w_salaries_dimensions.groupby(['soc_4_digit_name']).agg({'FLEX_HOURS': sum, 'FLEX_LOC': sum,'L&D': sum, 'CAREER': sum,'id': 'size', 'hourly_wage': 'median'}).reset_index()

for dim in ['FLEX_HOURS', 'FLEX_LOC','L&D', 'CAREER']:
    prop_table[f'{dim}_perc'] = (prop_table[dim] / prop_table['id']) * 100

prop_table

In [ ]:
prop_table[['soc_4_digit_name', 'id', 'hourly_wage', 'CAREER_perc', 'FLEX_HOURS_perc','FLEX_LOC_perc', 'L&D_perc']].to_csv('outputs/sector_multi_comparison.csv')

In [ ]:
def stacked_bar(counts_df, x, y, colour):
    # Create a stacked bar chart
    fig = px.bar(
        counts_df,
        x=x, 
        y=y,
        color=colour,
        orientation='h',  # Horizontal bars
        # title='Proportion of Job Adverts Offering Flexible Location and Hours by SOC',
        labels={'Proportion': 'Proportion of Adverts', 'soc_4_digit_name': 'SOC 4-Digit Name'},
        barmode='stack'  # Stacked bars
    )
    
    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        yaxis={'categoryorder':'total ascending'},  # Sort bars by total proportion
    )

    return fig

def grouped_bar(counts_df, x, y, colour):
    # Create a grouped bar chart (side-by-side bars)
    fig = px.bar(
        counts_df,
        x=x, 
        y=y,
        color=colour,
        orientation='h',  # Horizontal bars
        # title='Proportion of Job Adverts Offering Flexible Location and Hours by SOC',
        labels={'Proportion': 'Proportion of Adverts', 'soc_4_digit_name': 'SOC 4-Digit Name'},
        barmode='group'  # Grouped bars (side by side)
    )
    
    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        yaxis={'categoryorder':'total ascending'},  # Sort bars by total proportion
    )

    return fig

In [ ]:
grouped_soc = sample_w_salaries_dimensions.groupby("soc_4_digit_name").agg(
    total_ads=("id", "count"),
    flex_loc_mentions=("FLEX_LOC", "sum"),
    flex_hours_mentions=("FLEX_HOURS", "sum")
).reset_index()

# Calculate proportion
grouped_soc["flex_loc_proportion"] = grouped_soc["flex_loc_mentions"] / grouped_soc["total_ads"]
grouped_soc["flex_hours_proportion"] = grouped_soc["flex_hours_mentions"] / grouped_soc["total_ads"]


melted_soc = grouped_soc.melt(id_vars=["soc_4_digit_name"], 
                              value_vars=["flex_loc_proportion", "flex_hours_proportion"],
                              var_name="Flexibility Type",
                              value_name="Proportion")

# Map better labels
melted_soc["Flexibility Type"] = melted_soc["Flexibility Type"].map({
    "flex_loc_proportion": "Flexible Location",
    "flex_hours_proportion": "Flexible Hours"
})

grouped_bar(melted_soc, x="Proportion", y="soc_4_digit_name", colour="Flexibility Type")


In [ ]:
# import plotly.graph_objects as go

# fig = go.Figure()

# # Add traces for counts
# for flex_type in melted_all["Flexibility Type"].unique():
#     fig.add_trace(go.Bar(
#         x=melted_all.loc[(melted_all["Flexibility Type"] == flex_type) & (melted_all["Metric"] == "Count"), "Value"],
#         y=melted_all.loc[(melted_all["Flexibility Type"] == flex_type) & (melted_all["Metric"] == "Count"), "soc_4_digit_name"],
#         name=flex_type,
#         orientation='h',
#         visible=True  # Initially show counts
#     ))

# # Add traces for proportions (initially hidden)
# for flex_type in melted_all["Flexibility Type"].unique():
#     fig.add_trace(go.Bar(
#         x=melted_all.loc[(melted_all["Flexibility Type"] == flex_type) & (melted_all["Metric"] == "Proportion"), "Value"],
#         y=melted_all.loc[(melted_all["Flexibility Type"] == flex_type) & (melted_all["Metric"] == "Proportion"), "soc_4_digit_name"],
#         name=flex_type,
#         orientation='h',
#         visible=False  # Initially hide proportions
#     ))

# # Create dropdown buttons to toggle between counts and proportions
# fig.update_layout(
#     updatemenus=[
#         {
#             "buttons": [
#                 {
#                     "label": "Show Counts",
#                     "method": "update",
#                     "args": [{"visible": [True, True, False, False]},  # Toggle traces
#                              {"title": "Number of Job Adverts Offering Flexible Location and Hours by SOC"}]
#                 },
#                 {
#                     "label": "Show Proportion",
#                     "method": "update",
#                     "args": [{"visible": [False, False, True, True]},  # Toggle traces
#                              {"title": "Proportion of Job Adverts Offering Flexible Location and Hours by SOC"}]
#                 }
#             ],
#             "direction": "down",
#             "showactive": True,
#         }
#     ],
#     barmode="group",  # Side-by-side bars
#     title="Number of Job Adverts Offering Flexible Location and Hours by SOC",
#     width=1000,
#     height=800,
#     yaxis={'categoryorder': 'total ascending'}
# )

# fig.show()

In [ ]:
grouped = sample_w_salaries_dimensions.groupby(["year", "soc_4_digit_name"]).agg(
    total_ads=("id", "count"),
    flex_loc_mentions=("FLEX_LOC", "sum"),
    flex_hours_mentions=("FLEX_HOURS", "sum")
).reset_index()

# Calculate proportion
grouped["flex_loc_proportion"] = grouped["flex_loc_mentions"] / grouped["total_ads"]
grouped["flex_hours_proportion"] = grouped["flex_hours_mentions"] / grouped["total_ads"]


In [ ]:
grouped

In [ ]:
grouped

In [ ]:

# Filter only 2021 and 2024 data
df_filtered = grouped[grouped["year"].isin([2021, 2024])]

# Pivot table to compare 2021 vs 2024 values
df_pivot = df_filtered.pivot(index="soc_4_digit_name", columns="year", values="flex_hours_proportion")

# Drop rows that don't have both years
df_pivot = df_pivot.dropna()

# Calculate change in flex_hours_proportion
df_pivot["change"] = df_pivot[2024] - df_pivot[2021]

# Sort by biggest increase
df_pivot_sorted = df_pivot.sort_values(by="change", ascending=False)

df_pivot_sorted = df_pivot_sorted.reset_index()

# Create the bar chart
fig = px.bar(df_pivot_sorted.reset_index(), 
             x="change", 
             y="soc_4_digit_name", 
             orientation="h",
             title="Change in Flex Hours Proportion (2021 to 2024)",
             labels={"change": "Change in Flex Hours Proportion", "soc_4_digit_name": "SOC 4-Digit Name"},
             color="change",  # Automatically color positive/negative bars
             color_continuous_scale=["red", "blue"]  # Red for decreases, blue for increases
)

# Add a vertical line at 0 for reference
fig.add_vline(x=0, line_dash="dash", line_color="black")

# Improve layout
fig.update_layout(
    width=1000,
    height=800,
    xaxis_title="Increase / Decrease in Flex Hours Proportion 2021-2024",
    yaxis_title=None,#"SOC 4-Digit Name",
    coloraxis_showscale=False  # Hide color scale
)

fig.show()

In [ ]:
# Melt dataframe for easier visualization
melted = grouped.melt(id_vars=["year", "soc_4_digit_name"], 
                      value_vars=["flex_loc_proportion", "flex_hours_proportion"],
                      var_name="Flexibility Type",
                      value_name="Proportion")

# Map better labels
melted["Flexibility Type"] = melted["Flexibility Type"].map({
    "flex_loc_proportion": "Flexible Location",
    "flex_hours_proportion": "Flexible Hours"
})

# Plot with Plotly
fig = px.line(melted, x="year", y="Proportion", 
              color="Flexibility Type",
              facet_col="soc_4_digit_name",  # Separate by SOC name
              facet_col_wrap=3,  # Arrange in multiple rows
              title="Proportion of Job Adverts Mentioning Flexibility Over Time",
              labels={"year": "Year", "Proportion": "% of ads"},
              markers=True, template="plotly_white")

# Adjust layout for better readability
fig.update_layout(
    height=1600, 
    width=1200,
    xaxis=dict(type="category")  # Ensure years are treated as discrete values
)

## Contract type

In [ ]:
contracts_full = get_contracts_full()
contract_df = get_contract_per_job()
contracts_complete = get_contracts_complete()

In [ ]:
contract_df['final_contract_type'].value_counts()

In [ ]:
# contracts_complete = contract_df[contract_df['final_contract_type'] != 'Unknown']
len(contracts_complete) / len(healthcare_sample) # We can get contract type for 20% of the sample

In [ ]:
contracts_salaries_soc_region = pd.merge(contracts_complete, healthcare_sample, on='id', how='inner')

In [ ]:
# Count occurrences of final_contract_type for each soc_4_digit_name
contract_counts = contracts_salaries_soc_region.groupby(['soc_4_digit_name', 'final_contract_type']).size().reset_index(name='count')

def stacked_bar(counts_df, x, y, colour):
    # Create a stacked bar chart
    fig = px.bar(
        counts_df,
        x=x, 
        y=y,
        color=colour,
        orientation='h',  # Horizontal bars
        # title='Distribution of Final Contract Types by SOC 4-Digit Name',
        # labels={'count': 'Count', 'soc_4_digit_name': 'SOC 4-Digit Name'},
        barmode='stack'  # Stacked bars
    )
    
    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        # xaxis_tickangle=-45,  # Rotate x-axis labels for readability)
    )

    return fig

stacked_bar(contract_counts, x='count', y='soc_4_digit_name', colour='final_contract_type')


In [ ]:
contract_counts['percentage'] = contract_counts.groupby('soc_4_digit_name')['count'].transform(lambda x: x / x.sum() * 100)

stacked_bar(contract_counts, x='percentage', y='soc_4_digit_name', colour='final_contract_type')